In [1]:
import requests, time, random, re, csv, os, json
from bs4 import BeautifulSoup

In [2]:
# Define a robust master dictionary of core tools for Data & AI roles
TECH_KEYWORDS = {
    "Languages": ["python", "r", "sql", "java", "scala", "julia", "c\\+\\+"],
    "BI Tools": ["power bi", "tableau", "looker", "qlik", "excel"],
    "Cloud Platforms": ["aws", "azure", "gcp", "snowflake", "bigquery", "databricks"],
    "Data Engineering": ["spark", "hadoop", "airflow", "dbt", "kafka", "docker", "kubernetes"],
    "Libraries/Frameworks": ["pandas", "numpy", "scikit-learn", "tensorflow", "pytorch"]
}

def extract_skills(description_text):
    if not description_text:
        return json.dumps([])
    matched_skills = []
    for category, skills in TECH_KEYWORDS.items():
        for skill in skills:
            pattern = rf'\b{skill}\b'
            if re.search(pattern, description_text):
                matched_skills.append({
                    "skill_name": skill.upper().replace("\\", ""), 
                    "skill_category": category
                })
    return json.dumps(matched_skills)

# --- ARCHITECTURAL UPGRADE: TARGET HUBS WITH METADATA PRESERVED ---
target_hubs = [
    {"city": "San Francisco", "country": "United States", "continent": "North America"},
    {"city": "New York", "country": "United States", "continent": "North America"},
    {"city": "Toronto", "country": "Canada", "continent": "North America"},
    {"city": "London", "country": "United Kingdom", "continent": "Europe"},
    {"city": "Amsterdam", "country": "Netherlands", "continent": "Europe"},
    {"city": "Berlin", "country": "Germany", "continent": "Europe"},
    {"city": "Bangalore", "country": "India", "continent": "Asia"},
    {"city": "Singapore", "country": "Singapore", "continent": "Asia"},
    {"city": "Sydney", "country": "Australia", "continent": "APAC"},
    {"city": "Sao Paulo", "country": "Brazil", "continent": "South America"},
    {"city": "Cairo", "country": "Egypt", "continent": "Africa"}
]

pages_to_scrape = 4  # 100 jobs per location loop
keywords_list = ["Data Analyst", "Machine Learning", "AI"]

headers_browser = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

# Added clean split tracking columns to eliminate hard parsing later
csv_headers = [
    "job_id", "job_title", "company", "location_raw", 
    "city_clean", "country_clean", "continent_clean",
    "salary_raw", "seniority_level", "required_skills"
]

filename = "data/raw/raw_data.csv"
os.makedirs(os.path.dirname(filename), exist_ok=True)
file_exists = os.path.isfile(filename)

print("Starting global job scrape operations...")

with open(filename, 'a', newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    if not file_exists:
        writer.writerow(csv_headers)
        
    for hub in target_hubs:
        current_city = hub["city"]
        current_country = hub["country"]
        current_continent = hub["continent"]
        
        print(f"Scraping Hub: {current_city}, {current_country}...")
        
        for keyword in keywords_list:
            for page in range(pages_to_scrape):
                start_val = page * 25
                target_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={keyword}&location={current_city}&start={start_val}"
                
                try:
                    response = requests.get(target_url, headers=headers_browser, timeout=10)
                    if response.status_code != 200:
                        print(f"  [Skipping] List page status code: {response.status_code}")
                        continue
                        
                    soup = BeautifulSoup(response.text, 'html.parser')
                    parent_job_list_items = soup.find_all('li')
                    
                    for job in parent_job_list_items:
                        # 1. Extract and clean Job ID
                        job_card = job.find('div', class_=re.compile(r'job-search-card'))
                        if not job_card:
                            job_card = job.find('div', class_=re.compile(r'base-card'))
                        
                        raw_job_id = job_card.get('data-entity-urn') if job_card else None
                        if not raw_job_id:
                            continue
                        clean_job_id = re.sub(r"urn:li:jobPosting:", "", raw_job_id)
                        
                        # 2. Extract Title and Company
                        title_tag = job.find('span', class_='sr-only')
                        job_title = title_tag.text.strip() if title_tag else keyword
                        
                        company_tag = job.find('a', class_="hidden-nested-link")
                        if not company_tag:
                            company_tag = job.find('h4', class_="base-search-card__subtitle")
                        company = company_tag.text.strip() if company_tag else "Unknown Company"
                        
                        location_tag = job.find('span', class_='job-search-card__location')
                        location_raw = location_tag.text.strip() if location_tag else current_city
                        
                        # --- DEFENSIVE DOWNSTREAM EXTRACTION ---
                        # Injecting adaptive back-off delay to stop anti-bot triggers 
                        time.sleep(random.uniform(1.2, 2.8))
                        
                        detail_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{clean_job_id}"
                        res = requests.get(detail_url, headers=headers_browser, timeout=10)
                        
                        if res.status_code == 429:
                            print("  [Warning] Facing 429 rate limits. Cool-down sleep triggered...")
                            time.sleep(random.uniform(15.0, 30.0))
                            res = requests.get(detail_url, headers=headers_browser, timeout=10)
                        
                        salary_text = "Not Mentioned"
                        seniority_level = "Not Applicable"
                        desc_div_text = ""
                        
                        if res.status_code == 200:
                            s = BeautifulSoup(res.text, 'html.parser')
                            
                            # Standardized salary block string extraction
                            salary_tag = s.find('div', class_="compensation__salary-range")
                            if salary_tag:
                                # Keep raw layout format intact for robust regex cleaning pipeline
                                salary_text = " ".join(salary_tag.text.split()) 
                            
                            seniority_tag = s.find('span', class_='description__job-criteria-text description__job-criteria-text--criteria')
                            if seniority_tag:
                                seniority_level = seniority_tag.text.strip()
                                
                            desc_div = s.find('div', class_='description__text--rich')
                            if desc_div:
                                desc_div_text = desc_div.text.lower()

                        # Write row directly mapping structural fields 
                        writer.writerow([
                            clean_job_id, job_title, company, location_raw,
                            current_city, current_country, current_continent,
                            salary_text, seniority_level, extract_skills(desc_div_text)
                        ])
                        
                except Exception as e:
                    print(f"  [Error encountered]: {e}")
                    time.sleep(5)
                    
            # Human behavioral sleep interval per cycle completion
            time.sleep(random.uniform(4.0, 8.5))

print("Optimization complete. Your structured global csv file is ready!")

Starting global job scrape operations...
Scraping Hub: San Francisco, United States...
Scraping Hub: New York, United States...
Scraping Hub: Toronto, Canada...
Scraping Hub: London, United Kingdom...
  [Error encountered]: HTTPSConnectionPool(host='www.linkedin.com', port=443): Max retries exceeded with url: /jobs-guest/jobs/api/jobPosting/4416891952 (Caused by NewConnectionError("HTTPSConnection(host='www.linkedin.com', port=443): Failed to establish a new connection: [WinError 10065] A socket operation was attempted to an unreachable host"))
  [Error encountered]: HTTPSConnectionPool(host='www.linkedin.com', port=443): Max retries exceeded with url: /jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=Machine%20Learning&location=London&start=0 (Caused by NameResolutionError("HTTPSConnection(host='www.linkedin.com', port=443): Failed to resolve 'www.linkedin.com' ([Errno 11001] getaddrinfo failed)"))
  [Error encountered]: HTTPSConnectionPool(host='www.linkedin.com', port=443): Ma